# Experiments 010 to 012: the full 1.5B mirror (Colab)

One session that gives 1.5b everything the other scales have: 010 is the
mc-sft exposure sweep (mirrors 007), 011 is warm-start mc-sft chained from the
009 seed-0 gsm8k adapter (mirrors 008), 012 is base-to-grpo (mirrors 005/004).

Expectations on record before the run: 010 lands near the .69 probe ceiling
with sub-1pt seed spread, at or above grpo's .655; 011 changes nothing vs 010
(the warm start added nothing at either other scale); 012 is the open one,
0.5b collapsed on format without the warm start and 3b did not, so 1.5b
locates the threshold.

Order is cheap-first: sweeps and gen evals (~3h), then the 012 grpo run
(~5h). ~8h total, probably two sessions, everything resume-safe.

## 0. Compute units and mount Drive

Fill `cu_before` now. The open record is saved straight after Drive mounts,
so a disconnected session can be closed when the next runtime starts.

In [11]:
# note the balance before anything burns units, then save it after drive mounts
import glob, json, os, time
session_id = globals().get('session_id', time.strftime('%Y%m%d_%H%M%S'))
cu_before = 276.19   # e.g. 87.3
print('session', session_id, '| cu_before =', cu_before if cu_before is not None else 'FILL ME')

from google.colab import drive
drive.mount('/content/drive')
ROOT = '/content/drive/MyDrive/inm363'
RESULTS = os.path.join(ROOT, 'results')
OPEN_UNITS = os.path.join(RESULTS, 'open_cu_sessions')
UNITS_LOG = os.path.join(RESULTS, 'colab_units.jsonl')
os.makedirs(OPEN_UNITS, exist_ok=True)

# close unfinished session with the balance captured at this runtime start
old_sessions = [p for p in glob.glob(os.path.join(OPEN_UNITS, '*.json'))
                if not p.endswith(f'{session_id}.json')]
open_path = os.path.join(OPEN_UNITS, f'{session_id}.json')
if cu_before is None:
    print('reminder: fill cu_before, then rerun this cell to save the session')
else:
    for old_path in old_sessions:
        with open(old_path) as f:
            old = json.load(f)
        old['ts_end'] = time.strftime('%Y-%m-%d %H:%M')
        old['colab_cu_after'] = cu_before
        old['colab_cu_used'] = round(old['colab_cu_before'] - cu_before, 2)
        old['closed_on_resume'] = True
        with open(UNITS_LOG, 'a') as f:
            f.write(json.dumps(old) + '\n')
        os.remove(old_path)
        print('closed previous session:', old['session_id'])

    if not os.path.exists(open_path):
        start = {'session_id': session_id, 'ts_start': time.strftime('%Y-%m-%d %H:%M'),
                 'notebook': '010_012_colab_1p5b_mirror', 'colab_cu_before': cu_before}
        with open(open_path, 'w') as f:
            json.dump(start, f, indent=2)
        print('saved open session:', session_id)
print('drive mounted ->', ROOT)


session 20260721_103214 | cu_before = 276.19
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
closed previous session: 20260720_213054
saved open session: 20260721_103214
drive mounted -> /content/drive/MyDrive/inm363


## 1. Check the GPU

In [12]:
import subprocess
# reportable cells run on a100/bf16 only (runtime consistency is part of the
# protocol), so stop here instead of finding out five hours into training
gpu = subprocess.run(['nvidia-smi', '-L'], capture_output=True, text=True).stdout
print(gpu.strip() or 'no gpu visible')
assert 'A100' in gpu, 'wrong runtime: pick a100 under runtime > change runtime type'

GPU 0: NVIDIA A100-SXM4-80GB (UUID: GPU-caf2c5a2-4b2d-a557-7fe0-f952c3ffa134)


## 2. Clone or update the repo (automatic)

In [13]:
import base64
import getpass
import glob
import os
from pathlib import Path
import subprocess

REPO_SLUG = 'hastov-lab/rl-grpo-or-sft-for-llms'
repo_url = f'https://github.com/{REPO_SLUG}.git'
repo_root = Path('/content') / REPO_SLUG.split('/')[-1]


def repo_git(*args):
    git = ['git']
    env = None
    try:
        from google.colab import userdata
    except ImportError:
        userdata = None

    # Private forks can use a read token from Colab Secrets.
    if userdata is not None and REPO_SLUG != 'hastov-lab/rl-grpo-or-sft-for-llms':
        try:
            token = userdata.get('GH_TOKEN')
        except Exception:
            token = None
        if isinstance(token, dict):
            token = token.get('value')
        if not isinstance(token, str) or not token.strip():
            token = getpass.getpass('GitHub token: ')
        if not token.strip():
            raise RuntimeError('This private fork requires a GitHub token with read access.')

        # Keep the token out of the command line and the saved remote URL.
        auth = base64.b64encode(f'x-access-token:{token.strip()}'.encode()).decode()
        env = dict(os.environ, REPO_GIT_AUTH=f'Authorization: Basic {auth}')
        git.append('--config-env=http.https://github.com/.extraHeader=REPO_GIT_AUTH')

    subprocess.run([*git, *args], env=env, check=True)


if repo_root.exists():
    if not (repo_root / '.git').exists():
        raise RuntimeError(f'Expected a Git checkout at {repo_root}.')
    origin = subprocess.check_output(
        ['git', '-C', str(repo_root), 'config', '--local', '--get', 'remote.origin.url'], text=True
    ).strip()
    if origin != repo_url:
        raise RuntimeError(f'The checkout at {repo_root} belongs to a different repository.')
    # a resumed runtime already has the clone, in that case pull instead of failing
    # and silently running whatever old code is lying around
    branch = subprocess.check_output(
        ['git', '-C', str(repo_root), 'branch', '--show-current'], text=True
    ).strip()
    if branch != 'main':
        raise RuntimeError(f'Expected the main branch at {repo_root}.')
    repo_git('-C', str(repo_root), 'pull', '--ff-only', 'origin', 'main')
else:
    repo_git('clone', '--branch', 'main', '--single-branch', '--depth', '1', repo_url, str(repo_root))

# Use the scripts from this checkout even if another experiment is open.
code_root = repo_root / '001_Code'
if not (code_root / 'scripts' / 'run.py').is_file():
    raise RuntimeError(f'Experiment scripts are missing from {code_root}.')
os.chdir(code_root)
print('working in:', os.getcwd())

remote: Enumerating objects: 29, done.
remote: Counting objects: 100% (29/29), done.
remote: Compressing objects: 100% (3/3), done.
remote: Total 18 (delta 15), reused 18 (delta 15), pack-reused 0 (from 0)
Unpacking objects: 100% (18/18), 8.85 KiB | 201.00 KiB/s, done.
From https://github.com/oleh-hastov/INM363-IndividualProject-Dissertation
   7f2499d..96ae500  main       -> origin/main
Updating 7f2499d..96ae500
Fast-forward
 001_Code/configs/010_p2_colab_b15.yaml             | 52 +++++++++++++++++++
 001_Code/configs/010_p32_colab_b15.yaml            | 52 +++++++++++++++++++
 001_Code/configs/010_p8_colab_b15.yaml             | 52 +++++++++++++++++++
 001_Code/configs/011_wsft_colab_b15.yaml           | 53 ++++++++++++++++++++
 .../012_colab_full_b15_base_grpo_seed0.yaml        | 58 ++++++++++++++++++++++
 001_Code/notebooks/009_2_analysis_plots.ipynb      |  4 +-
 001_Code/notebooks/010_012_colab_1p5b_mirror.ipynb |  1 +
 001_Code/results/compute_ledger.jsonl              |  1 -
 00

## 3. Install dependencies

In [14]:
!pip install -q trl==1.7.1 transformers==5.5.4 peft==0.19.1 accelerate==1.14.0 datasets==4.8.4 wandb pyyaml
# Colab preinstalls a torchao that peft 0.19.1 chokes on inside its lora
# dispatcher. No quantization here, so just drop it
!pip uninstall -q -y torchao
print('deps installed')


deps installed


## 4. W&B key (automatic)

Reminder: colab secrets are granted per notebook, switch on access for
`WANDB_API_KEY` under the key icon before running. This key is only used for experiment tracking.


In [15]:
import getpass
import os

wkey = None
try:
    from google.colab import userdata
    wkey = userdata.get('WANDB_API_KEY')
except Exception:
    wkey = None

if not wkey:
    wkey = getpass.getpass('W&B API key: ').strip()
if not wkey:
    raise RuntimeError('WANDB_API_KEY is required')

os.environ['WANDB_API_KEY'] = wkey
with open('wandb_api_key.txt', 'w') as f:
    f.write(wkey.strip())
print('W&B key loaded:', os.path.exists('wandb_api_key.txt'))


W&B key loaded: True


## 5. Check the Drive record

In [16]:
if os.path.exists(open_path):
    print('open compute-unit record:', open_path)
else:
    print('open record missing, rerun section 0 with cu_before filled')


open compute-unit record: /content/drive/MyDrive/inm363/results/open_cu_sessions/20260721_103214.json


## 6. Build the data splits

In [17]:
!python scripts/build_data.py


dropped 1 grpo prompt(s) that also sit in the test split:
  [high_school_psychology] 'Answer the multiple choice question. Think through it step by step inside <think></think> tags, then'...
sft rows:  7473
grpo rows: 245  by tier: {'easy': 110, 'medium': 82, 'hard': 53}
eval rows: 1884  by tier: {'easy': 879, 'medium': 609, 'hard': 396}
no leakage, three splits ready


## 7. Preflight: the 009 stage-1 adapter must be on Drive

011 chains from the 009 seed-0 gsm8k-sft checkpoint, check it before burning
hours. 010 and 012 start from the base and need nothing.

In [18]:
import os
D = '/content/drive/MyDrive/inm363'
rel = 'checkpoints/009_COLAB_FULL_B15_sft_grpo_seed0/sft'
for fname in ('adapter_model.safetensors', 'adapter_config.json', 'adapter_base.json'):
    p = os.path.join(D, rel, fname)
    assert os.path.exists(p), f'stage-1 file missing: {p}'
print('warm start present and complete')

warm start present and complete


## 8. Wiring check

In [19]:
!python scripts/run.py --config configs/010_p8_colab_b15.yaml

env=COLAB mode=FULL scale=B15 condition=SFT_ONLY
model=Qwen/Qwen2.5-1.5B device=cuda stages=['base_eval', 'sft', 'sft_eval']
steps=123 N=8 seed=0
trackers=['tensorboard', 'wandb']
results -> /content/drive/MyDrive/inm363/results/010_p8_COLAB_FULL_B15_sft_only_seed0/  checkpoints -> /content/drive/MyDrive/inm363/checkpoints/010_p8_COLAB_FULL_B15_sft_only_seed0/
local smoke wiring ok


## 9. Experiment 010: MC-SFT sweep, probe-only

9 small runs: 3 exposures x 3 seeds, minutes each.

**If colab disconnects:** re-run the cell, finished runs skip in seconds.

In [20]:
# python loop with ! calls so the logs stream live, exit code checked so a
# failed run stops the sweep instead of hiding behind the next one
for c in ('010_p2_colab_b15', '010_p8_colab_b15', '010_p32_colab_b15'):
    for s in (0, 1, 2):
        print(f'===== {c} seed {s} =====')
        !PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True python scripts/run.py --config configs/{c}.yaml --seed {s} --stage all
        if _exit_code != 0:
            raise SystemExit(f'{c} seed {s} failed with exit code {_exit_code}')

===== 010_p2_colab_b15 seed 0 =====
seed overridden to 0
2026-07-21 10:44:43,919 INFO using wandb key from env
2026-07-21 10:44:43,920 INFO run 010_p2_COLAB_FULL_B15_sft_only_seed0 device=cuda stages=['base_eval', 'sft', 'sft_eval'] trackers=['tensorboard', 'wandb'] out=/content/drive/MyDrive/inm363/results/010_p2_COLAB_FULL_B15_sft_only_seed0
2026-07-21 10:44:43,920 INFO base_eval skipped, probe-only run
config.json: 100% 684/684 [00:00<00:00, 3.38MB/s]
tokenizer_config.json: 100% 7.23k/7.23k [00:00<00:00, 23.5MB/s]
vocab.json: 100% 2.78M/2.78M [00:00<00:00, 108MB/s]
merges.txt: 100% 1.67M/1.67M [00:00<00:00, 115MB/s]
tokenizer.json: 100% 7.03M/7.03M [00:00<00:00, 136MB/s]

model.safetensors: downloading bytes:   8% 262M/3.09G [00:02<00:14, 199MB/s, 21.6MB/s  ]
model.safetensors: downloading bytes:  22% 687M/3.09G [00:04<00:06, 360MB/s, 55.1MB/s  ]
model.safetensors: downloading bytes:  36% 1.11G/3.09G [00:04<00:03, 547MB/s, 85.2MB/s  ]
model.safetensors: downloading bytes:  81% 2.51G

## 10. Experiment 011: warm-start MC-SFT, probe-only

3 runs at 8 passes. The first merges the stage-1 adapter into the base once,
seeds 1 and 2 reuse the shared merged model.

In [21]:
for s in (0, 1, 2):
    print(f'===== 011_wsft seed {s} =====')
    !PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True python scripts/run.py --config configs/011_wsft_colab_b15.yaml --seed {s} --stage all
    if _exit_code != 0:
        raise SystemExit(f'011_wsft seed {s} failed with exit code {_exit_code}')

===== 011_wsft seed 0 =====
seed overridden to 0
2026-07-21 11:31:34,860 INFO using wandb key from env
2026-07-21 11:31:34,861 INFO run 011_wsft_COLAB_FULL_B15_sft_only_seed0 device=cuda stages=['base_eval', 'sft', 'sft_eval'] trackers=['tensorboard', 'wandb'] out=/content/drive/MyDrive/inm363/results/011_wsft_COLAB_FULL_B15_sft_only_seed0
2026-07-21 11:31:34,861 INFO base_eval skipped, probe-only run
Loading weights: 100% 338/338 [00:00<00:00, 4751.27it/s]
base logprob (1884 items): 100% 118/118 [00:06<00:00, 17.50it/s, acc 0.692]
2026-07-21 11:32:02,133 INFO base_logprob_eval done in 27s | elapsed 27s
Loading weights: 100% 338/338 [00:00<00:00, 5377.80it/s]
Writing model shards: 100% 1/1 [00:04<00:00,  4.20s/it]
Loading weights: 100% 338/338 [00:01<00:00, 181.14it/s]
Adding EOS to train dataset: 100% 245/245 [00:00<00:00, 22580.25 examples/s]
Tokenizing train dataset: 100% 245/245 [00:00<00:00, 1041.10 examples/s]
Building labels for train dataset: 100% 245/245 [00:00<00:00, 3840.68 

## 11. Generative evals for 010 and 011

Mirrors the 007/008 final state: seed 0 at every exposure plus all three seeds
at p8, and all three 011 seeds. 8 evals, fast with the short mc-sft outputs.

In [22]:
D = '/content/drive/MyDrive/inm363'
jobs = [('010_p2_COLAB_FULL_B15_sft_only_seed0', 'configs/010_p2_colab_b15.yaml'),
        ('010_p32_COLAB_FULL_B15_sft_only_seed0', 'configs/010_p32_colab_b15.yaml')]
for s in (0, 1, 2):
    jobs.append((f'010_p8_COLAB_FULL_B15_sft_only_seed{s}', 'configs/010_p8_colab_b15.yaml'))
    jobs.append((f'011_wsft_COLAB_FULL_B15_sft_only_seed{s}', 'configs/011_wsft_colab_b15.yaml'))
for tag, cfg in jobs:
    print(f'===== gen eval {tag} =====')
    seed = tag.rsplit('seed', 1)[1]
    !python scripts/eval_adapter.py --config {cfg} --adapter {D}/checkpoints/{tag}/sft --condition sft --out {D}/results/{tag} --seed {seed}
    if _exit_code != 0:
        raise SystemExit(f'{tag} eval failed with exit code {_exit_code}')

===== gen eval 010_p2_COLAB_FULL_B15_sft_only_seed0 =====
Loading weights: 100% 338/338 [00:00<00:00, 6195.10it/s]
sft eval (1884 items): 100% 118/118 [04:43<00:00,  2.40s/it, acc 0.670 fmt 0.997]
sft probe already done, skipping
summary completed with the new evals
gen per-tier: {'easy': 0.7940841865756542, 'medium': 0.6600985221674877, 'hard': 0.4090909090909091}
probe per-tier: {'easy': 0.8191126279863481, 'medium': 0.6995073891625616, 'hard': 0.41919191919191917}
===== gen eval 010_p32_COLAB_FULL_B15_sft_only_seed0 =====
Loading weights: 100% 338/338 [00:00<00:00, 5623.57it/s]
sft eval (1884 items): 100% 118/118 [04:42<00:00,  2.40s/it, acc 0.695 fmt 0.995]
sft probe already done, skipping
summary completed with the new evals
gen per-tier: {'easy': 0.8077360637087599, 'medium': 0.6929392446633826, 'hard': 0.44696969696969696}
probe per-tier: {'easy': 0.820250284414107, 'medium': 0.6863711001642037, 'hard': 0.4595959595959596}
===== gen eval 010_p8_COLAB_FULL_B15_sft_only_seed0 ====

## 12. Experiment 012: base-to-GRPO, full run

The expensive one, ~5h. GRPO straight on the pretrained 1.5b, frozen protocol,
seed 0, full generative evals inline.

In [23]:
!PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True python scripts/run.py --config configs/012_colab_full_b15_base_grpo_seed0.yaml --stage all
if _exit_code != 0:
    raise SystemExit(f'012 base-grpo failed with exit code {_exit_code}')

Streaming output truncated to the last 5000 lines.
│ │               │ resource      │               │              │           │ │
│ │               │ availability  │               │              │           │ │
│ │               │ and           │               │              │           │ │
│ │               │ confidential… │               │              │           │ │
│ │               │ policies, and │               │              │           │ │
│ │               │ major         │               │              │           │ │
│ │               │ projects in   │               │              │           │ │
│ │               │ evolution and │               │              │           │ │
│ │               │ genomics      │               │              │           │ │
│ │               │ based on      │               │              │           │ │
│ │               │ molecular     │               │              │           │ │
│ │               │ genetics.     │               │       

## 13. Peek at the results

In [24]:
import json, os
D = '/content/drive/MyDrive/inm363'
tags = [f'010_{p}_COLAB_FULL_B15_sft_only_seed{s}' for p in ('p2', 'p8', 'p32') for s in (0, 1, 2)]
tags += [f'011_wsft_COLAB_FULL_B15_sft_only_seed{s}' for s in (0, 1, 2)]
tags += ['012_COLAB_FULL_B15_base_grpo_seed0']
for tag in tags:
    d = os.path.join(D, 'results', tag)
    kind = 'grpo' if '012' in tag else 'sft'
    lp_f = os.path.join(d, f'{kind}_logprob.json')
    ev_f = os.path.join(d, f'{kind}_eval.json')
    lp = json.load(open(lp_f))['per_tier_acc'] if os.path.exists(lp_f) else None
    ev = json.load(open(ev_f))['per_tier_acc'] if os.path.exists(ev_f) else None
    fmt = lambda x: {k: round(v, 3) for k, v in x.items()} if x else '-'
    print(f'{tag}  probe {fmt(lp)}  gen {fmt(ev)}')

010_p2_COLAB_FULL_B15_sft_only_seed0  probe {'easy': 0.819, 'medium': 0.7, 'hard': 0.419}  gen {'easy': 0.794, 'medium': 0.66, 'hard': 0.409}
010_p2_COLAB_FULL_B15_sft_only_seed1  probe {'easy': 0.818, 'medium': 0.706, 'hard': 0.455}  gen -
010_p2_COLAB_FULL_B15_sft_only_seed2  probe {'easy': 0.82, 'medium': 0.701, 'hard': 0.427}  gen -
010_p8_COLAB_FULL_B15_sft_only_seed0  probe {'easy': 0.824, 'medium': 0.686, 'hard': 0.414}  gen {'easy': 0.804, 'medium': 0.701, 'hard': 0.427}
010_p8_COLAB_FULL_B15_sft_only_seed1  probe {'easy': 0.821, 'medium': 0.695, 'hard': 0.429}  gen {'easy': 0.8, 'medium': 0.688, 'hard': 0.452}
010_p8_COLAB_FULL_B15_sft_only_seed2  probe {'easy': 0.823, 'medium': 0.696, 'hard': 0.412}  gen {'easy': 0.795, 'medium': 0.693, 'hard': 0.465}
010_p32_COLAB_FULL_B15_sft_only_seed0  probe {'easy': 0.82, 'medium': 0.686, 'hard': 0.46}  gen {'easy': 0.808, 'medium': 0.693, 'hard': 0.447}
010_p32_COLAB_FULL_B15_sft_only_seed1  probe {'easy': 0.82, 'medium': 0.688, 'hard':

## 14. Close the units record

In [25]:
# closing balance, completes the open record for this session
import json, os, time
cu_after = 206.23   # fill from the resources panel again
if cu_after is None:
    print('reminder: fill cu_after for the record')
elif not os.path.isdir('/content/drive/MyDrive'):
    print('drive not mounted, run the mount cell first')
else:
    os.makedirs(OPEN_UNITS, exist_ok=True)
    open_path = os.path.join(OPEN_UNITS, f'{session_id}.json')
    if not os.path.exists(open_path):
        print('no open record for this session, it may already be closed')
    else:
        with open(open_path) as f:
            rec = json.load(f)
        rec['ts_end'] = time.strftime('%Y-%m-%d %H:%M')
        rec['colab_cu_after'] = cu_after
        rec['colab_cu_used'] = round(rec['colab_cu_before'] - cu_after, 2)
        rec['closed_on_resume'] = False
        with open(UNITS_LOG, 'a') as f:
            f.write(json.dumps(rec) + '\n')
        os.remove(open_path)
        print('logged:', rec)

reminder: fill cu_after for the record
